# RevisitingCIL for IP102 — Kaggle Notebook

**Input bắt buộc (chỉ 1):**

- **IP102 dataset** — folder chứa `train.json`/`test.json`/`val.json` + `VOC2007/VOC2007/JPEGImages` (dataset bạn vẫn dùng cho iCaRL)

Code được **clone tự động** từ `https://github.com/nta2112/RevisitingCIL-for-IP102` → cần bật **Internet** (Settings → Internet).

**Tuỳ biến qua env var (bật Settings → Notebook options → Environment variables):**

- `IP102_MODEL`: `simplecil` (mặc định) hoặc `aper_finetune`
- `IP102_MEMORY_SIZE`: tổng exemplar (mặc định 2000)
- `IP102_CODE_REPO`: URL repo khác nếu muốn

**2 GPU (khuyến nghị):** chọn accelerator **GPU T4 x2** — code đã an toàn multi-GPU (`drop_last` + batch chia hết theo số GPU).

**Chạy:** cell TEST NHANH 1 task mặc định active; muốn chạy đủ 4 task thì comment dòng test và uncomment dòng full (xem cell **Chạy**).

**Kết quả:** `/kaggle/working/logs/simplecil/ip102/...` — `results.csv`, `history.json`, `*.log`

In [ ]:
import os, sys, torch, timm
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())
print('timm', timm.__version__)

In [ ]:
import os, shutil, subprocess

REPO_URL = os.environ.get('IP102_CODE_REPO', 'https://github.com/nta2112/RevisitingCIL-for-IP102.git')
WORK_CODE = '/kaggle/working/RevisitingCIL'

def find_dir_with_file(base, fname, maxdepth=10):
    base = os.path.abspath(base)
    for dp, dn, fn in os.walk(base):
        depth = dp[len(base):].count(os.sep)
        if depth > maxdepth:
            dn[:] = []
            continue
        if fname in fn:
            return dp
    return None

if os.path.exists(WORK_CODE):
    print('Da co code, bo qua clone:', WORK_CODE)
elif subprocess.call(['git', 'clone', '--depth', '1', REPO_URL, WORK_CODE]) == 0:
    print('Cloned:', REPO_URL, '->', WORK_CODE)
else:
    CODE_SRC = find_dir_with_file('/kaggle/input', 'main.py')
    assert CODE_SRC, 'git clone that bai va khong tim thay code trong /kaggle/input'
    print('Clone that bai, dung code tu Input:', CODE_SRC)
    shutil.copytree(CODE_SRC, WORK_CODE, ignore=shutil.ignore_patterns('*.pyc', '__pycache__', '.git'))

assert os.path.exists(os.path.join(WORK_CODE, 'main.py')), 'Code khong hop le'
print('WORK_CODE:', WORK_CODE)

In [ ]:
import os

def find_dir_with_file(base, fname, maxdepth=10):
    base = os.path.abspath(base)
    for dp, dn, fn in os.walk(base):
        depth = dp[len(base):].count(os.sep)
        if depth > maxdepth:
            dn[:] = []
            continue
        if fname in fn:
            return dp
    return None

DATA_ROOT = os.environ.get('IP102_DATA_ROOT') or find_dir_with_file('/kaggle/input', 'train.json')
print('DATA_ROOT:', DATA_ROOT)
assert DATA_ROOT and os.path.exists(os.path.join(DATA_ROOT, 'train.json')), 'Khong tim thay IP102 dataset'

In [ ]:
import json, os, subprocess, sys

MODEL = os.environ.get('IP102_MODEL', 'simplecil').strip().lower()
MEMORY_SIZE = int(os.environ.get('IP102_MEMORY_SIZE', 2000))

CONFIG_REL = {
    'simplecil':     'exps/simplecil/ip102_7_6_6_6_vit-b_simplecil.json',
    'aper_finetune': 'exps/aper_finetune/ip102_7_6_6_6_vit-b_finetune.json',
}[MODEL]

def run_train(max_tasks=0):
    cfg = json.load(open(os.path.join(WORK_CODE, CONFIG_REL), encoding='utf-8'))
    cfg['memory_size'] = MEMORY_SIZE
    # 2-GPU-safe: dataloader drop_last + batch chia het theo so GPU
    cfg['device'] = [str(i) for i in range(torch.cuda.device_count())]
    if max_tasks > 0:
        n = min(max_tasks, 25 // 7 + 1)
        if n <= 1:
            cfg['increment'] = 25 - 7
        else:
            cfg['increment'] = -(-(25 - 7) // (n - 1))
    cfg_path = '/kaggle/working/config_tmp.json'
    json.dump(cfg, open(cfg_path, 'w', encoding='utf-8'), indent=2)
    print('model:', MODEL, '| devices:', cfg['device'], '| batch_size:', cfg['batch_size'], '| memory_size:', cfg['memory_size'])
    print('max_tasks:', max_tasks or 'all (4 tasks: 7,6,6,6)')
    print('config:', cfg_path)
    print('running main.py ...')
    sys.stdout.flush()
    ret = subprocess.call([sys.executable, 'main.py', '--config', cfg_path], cwd=WORK_CODE)
    print('main.py return code:', ret)
    return ret

## Chạy

- **TEST NHANH 1 task (13 lớp)** → chạy cell code ngay dưới (mặc định active).
- **CHẠY ĐỦ 4 TASK (7,6,6,6)** → comment dòng `run_train(max_tasks=1)` và uncomment dòng `run_train(max_tasks=0)`.

2 GPU T4: chọn accelerator **GPU T4 x2** (Settings → Accelerator). Config tự bật `device ["0","1"]`; dataloader đã sửa `drop_last` + batch chia hết theo số GPU.

In [ ]:
# ===== CHẠY ĐỦ 4 TASK (7,6,6,6): uncomment dòng dưới, comment dòng test =====
# run_train(max_tasks=0)

# ===== TEST NHANH 1 TASK (13 lớp) =====
run_train(max_tasks=1)


In [ ]:
import glob, os
import pandas as pd

csvs = glob.glob('/kaggle/working/logs/**/results.csv', recursive=True)
if not csvs:
    print('Chua co results.csv (train chua xong hoac that bai)')
else:
    for p in sorted(csvs):
        print('===', p)
        try:
            print(pd.read_csv(p).to_string(index=False))
        except Exception as e:
            print('khong doc duoc:', e)